# Build Processed Real Video Dataset

## 00 Runtime Identity And User Config (`00_runtime_identity_and_user_config`)

In [ ]:
from pathlib import Path
import json

from paper_workflow.notebook_utils import processed_real_video_dataset_workflow as dataset_workflow

WORKFLOW_KEY = 'processed_dataset_build'
RAW_DATASET_DOWNLOAD_MANIFEST_PATH = Path('/content/drive/MyDrive/Datasets/raw_dataset_download_manifest.json')
RAW_DATASET_KEY = 'davis_2017'
RAW_ARCHIVE_PATH = Path('/content/drive/MyDrive/Datasets/DAVIS_2017/raw/DAVIS-2017-trainval-480p.zip')
DRIVE_PROCESSED_ROOT = Path('/content/drive/MyDrive/TSTW/datasets/processed')
LOCAL_WORKSPACE_ROOT = Path('/content/TSTW_runtime/raw_workspaces/build_processed_real_video_dataset')
PROCESSED_DATASET_KEY = 'real_video_vae_latent_probe__davis2017_trainval480p__256x256__32f__8fps__freeze001'
PROCESSED_DATASET_ROOT = DRIVE_PROCESSED_ROOT / PROCESSED_DATASET_KEY
PROCESSED_DATASET_MANIFEST = PROCESSED_DATASET_ROOT / 'dataset_manifest.json'
PROCESSED_DATASET_SUMMARY = PROCESSED_DATASET_ROOT / 'processed_dataset_summary.json'
PROCESSED_DATASET_CHECKS = PROCESSED_DATASET_ROOT / 'checks' / 'processed_dataset_checks.json'

## 01 Mount Google Drive (`01_mount_google_drive`)

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

## 02 Validate Raw Dataset Registry (`02_validate_raw_dataset_registry`)

In [ ]:
if not RAW_DATASET_DOWNLOAD_MANIFEST_PATH.exists():
    raise FileNotFoundError(RAW_DATASET_DOWNLOAD_MANIFEST_PATH)
raw_dataset_registry = json.loads(RAW_DATASET_DOWNLOAD_MANIFEST_PATH.read_text(encoding='utf-8'))
print({'raw_dataset_download_manifest.json': str(RAW_DATASET_DOWNLOAD_MANIFEST_PATH)})

## 03 Select Raw Dataset Source (`03_select_raw_dataset_source`)

In [ ]:
raw_dataset_source = raw_dataset_registry.get('datasets', {}).get(RAW_DATASET_KEY, {})
print({
    'raw_dataset_key': RAW_DATASET_KEY,
    'raw_archive_path': str(RAW_ARCHIVE_PATH),
    'registry_entry_present': bool(raw_dataset_source),
})

## 04 Prepare Local Dataset Workspace (`04_prepare_local_dataset_workspace`)

In [ ]:
LOCAL_WORKSPACE_ROOT.mkdir(parents=True, exist_ok=True)
print({'local_workspace_root': str(LOCAL_WORKSPACE_ROOT)})

## 05 Extract Raw Dataset Archive (`05_extract_raw_dataset_archive`)

In [ ]:
print({
    'raw_archive_path': str(RAW_ARCHIVE_PATH),
    'extract_workspace': str(LOCAL_WORKSPACE_ROOT),
})

## 06 Build Processed Video Clips (`06_build_processed_video_clips`)

In [ ]:
processed_dataset_handoff = dataset_workflow.build_processed_dataset_handoff(
    raw_dataset_download_manifest_path=RAW_DATASET_DOWNLOAD_MANIFEST_PATH,
    raw_dataset_key=RAW_DATASET_KEY,
    raw_archive_path=RAW_ARCHIVE_PATH,
    processed_dataset_root=PROCESSED_DATASET_ROOT,
    processed_dataset_key=PROCESSED_DATASET_KEY,
    local_workspace_root=LOCAL_WORKSPACE_ROOT,
    clean_workspace=False,
)
print(processed_dataset_handoff)

## 07 Write Processed Dataset Manifest (`07_write_processed_dataset_manifest`)

In [ ]:
processed_dataset_manifest = json.loads(PROCESSED_DATASET_MANIFEST.read_text(encoding='utf-8'))
print({
    'dataset_name': processed_dataset_manifest['dataset_name'],
    'PROCESSED_DATASET_MANIFEST': str(PROCESSED_DATASET_MANIFEST),
})

## 08 Validate Processed Dataset (`08_validate_processed_dataset`)

In [ ]:
processed_dataset_checks = json.loads(PROCESSED_DATASET_CHECKS.read_text(encoding='utf-8'))
if not processed_dataset_checks.get('status'):
    raise RuntimeError(processed_dataset_checks)
print({'processed_dataset_checks.json': str(PROCESSED_DATASET_CHECKS), 'status': processed_dataset_checks['status']})

## 09 Register Processed Dataset (`09_register_processed_dataset`)

In [ ]:
print({'dataset_registry.json': processed_dataset_handoff['dataset_registry.json']})

## 10 Print Processed Dataset Handoff (`10_print_processed_dataset_handoff`)

In [ ]:
handoff_summary = {
    'PROCESSED_DATASET_KEY': PROCESSED_DATASET_KEY,
    'PROCESSED_DATASET_ROOT': str(PROCESSED_DATASET_ROOT),
    'PROCESSED_DATASET_MANIFEST': str(PROCESSED_DATASET_MANIFEST),
    'processed_dataset_summary.json': processed_dataset_handoff['processed_dataset_summary.json'],
    'processed_dataset_checks.json': processed_dataset_handoff['processed_dataset_checks.json'],
}
print(handoff_summary)